# Deep Agents - Subagents & Workflow

## 🎯 학습 목표
- Subagent의 개념과 사용 이유 이해하기
- Dictionary-based와 CompiledSubAgent 방식 배우기
- 복잡한 작업을 모듈화하고 구조화하는 방법 익히기
- 여러 Subagent가 협력하는 워크플로우 설계하기


## 1. Subagent란 무엇인가?

### 📌 Subagent의 필요성

복잡한 작업을 처리할 때, 하나의 Agent가 모든 것을 처리하면:
- 🚫 컨텍스트가 너무 커져서 관리하기 어려움
- 🚫 각 작업에 대한 전문성이 떨어짐
- 🚫 에러 발생 시 전체 시스템이 영향을 받음

### ✅ Subagent의 장점

1. **Context Isolation (컨텍스트 격리)**
   - 각 Subagent는 자신의 작업에만 집중
   - 메인 Agent의 컨텍스트가 깨끗하게 유지됨

2. **역할 분리 (Role Separation)**
   - 각 Subagent는 특정 작업에 전문화
   - 더 나은 성능과 정확도

3. **재사용성 (Reusability)**
   - Subagent를 여러 프로젝트에서 재사용 가능
   - 모듈화된 구조로 유지보수 용이


## 2. 환경 설정


In [ ]:
# 필요한 패키지 설치
%pip install -q deepagents tavily-python


In [ ]:
# 환경 변수 설정
import os
from getpass import getpass

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Tavily API Key: ")


## 3. Dictionary-based Subagent 정의하기

가장 간단한 방법은 딕셔너리로 Subagent를 정의하는 것입니다.


In [ ]:
from deepagents import create_agent

# 연구 전문 Subagent 정의
research_subagent = {
    "name": "researcher",
    "description": "웹에서 정보를 검색하고 조사하는 전문가입니다. 최신 정보가 필요할 때 이 agent를 호출하세요.",
    "system_prompt": """
    당신은 전문 연구원입니다.
    - 주어진 주제에 대해 철저히 조사합니다
    - 신뢰할 수 있는 출처를 찾습니다
    - 수집한 정보를 요약하여 반환합니다
    """
}

# 데이터 분석 전문 Subagent 정의
analyst_subagent = {
    "name": "analyst",
    "description": "데이터를 분석하고 인사이트를 도출하는 전문가입니다. 데이터 분석이 필요할 때 이 agent를 호출하세요.",
    "system_prompt": """
    당신은 데이터 분석 전문가입니다.
    - 제공된 데이터를 분석합니다
    - 패턴과 트렌드를 찾습니다
    - 실행 가능한 인사이트를 제공합니다
    """
}

# 보고서 작성 전문 Subagent 정의
writer_subagent = {
    "name": "writer",
    "description": "전문적인 보고서를 작성하는 전문가입니다. 문서 작성이 필요할 때 이 agent를 호출하세요.",
    "system_prompt": """
    당신은 전문 기술 작가입니다.
    - 명확하고 구조화된 문서를 작성합니다
    - 기술적 내용을 이해하기 쉽게 설명합니다
    - 적절한 형식과 스타일을 사용합니다
    """
}

print("✅ Subagent 정의 완료!")


In [ ]:
# Subagent들을 사용하는 Main Agent 생성
agent = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="""
    당신은 프로젝트 매니저입니다.
    복잡한 작업을 받으면 적절한 전문가에게 위임하세요:
    - 정보 조사가 필요하면 researcher에게
    - 데이터 분석이 필요하면 analyst에게  
    - 문서 작성이 필요하면 writer에게
    
    각 전문가의 결과를 취합하여 최종 결과를 제공하세요.
    """,
    subagents=[research_subagent, analyst_subagent, writer_subagent]
)

print("✅ Main Agent (with Subagents) 생성 완료!")


## 5. Subagent 워크플로우 실행하기


In [ ]:
# 복잡한 작업 요청
complex_task = """
다음 작업을 수행해주세요:

1. AI 에이전트의 2024년 주요 트렌드를 조사하세요
2. 수집한 정보에서 상위 3가지 트렌드를 분석하세요
3. 분석 결과를 바탕으로 'ai_trends_report.txt' 파일에 보고서를 작성하세요

보고서는 다음 섹션을 포함해야 합니다:
- 요약 (Executive Summary)
- 주요 트렌드 분석
- 시장 영향
- 결론 및 전망
"""

print("🔄 복잡한 작업 실행 중...\n")

response = agent.invoke(
    {"messages": [{"role": "user", "content": complex_task}]},
    config={"configurable": {"thread_id": "subagent-demo"}}
)

print("\n✅ 작업 완료!")
print(response["messages"][-1].content)


## 6. Subagent 실행 과정 관찰하기

스트리밍을 사용하여 Main Agent가 어떻게 Subagent들을 조율하는지 확인할 수 있습니다.


In [ ]:
# 스트리밍으로 실행 과정 확인
task = "LangChain의 최신 기능을 조사하고, 주요 개선사항 3가지를 분석한 후, 간단한 요약 보고서를 작성해주세요."

print("🔍 Subagent 워크플로우 실행 과정:\n")
print("="*60)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": task}]},
    config={"configurable": {"thread_id": "subagent-stream"}},
    stream_mode="updates"
):
    for node_name, node_output in chunk.items():
        print(f"\n📍 노드: {node_name}")
        
        # Subagent 호출 감지
        if "messages" in node_output:
            for msg in node_output["messages"]:
                # Tool call 확인 (Subagent 호출)
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    for tool_call in msg.tool_calls:
                        if tool_call.get('name') == 'delegate_to_subagent':
                            subagent_name = tool_call.get('args', {}).get('subagent_name', 'unknown')
                            print(f"   🤝 Subagent 호출: {subagent_name}")
                
                # 응답 내용
                if hasattr(msg, 'content') and msg.content:
                    content_preview = str(msg.content)[:150]
                    print(f"   💬 응답: {content_preview}...")

print("\n" + "="*60)


## 7. CompiledSubAgent 방식

더 고급 사용 사례를 위해 미리 컴파일된 Agent를 Subagent로 사용할 수 있습니다.


In [ ]:
# 독립적인 전문 Agent들을 먼저 생성
code_reviewer = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="""
    당신은 시니어 코드 리뷰어입니다.
    - 코드의 품질을 검토합니다
    - 버그와 개선점을 찾습니다
    - 구체적인 피드백을 제공합니다
    - 코딩 표준 준수 여부를 확인합니다
    """
)

security_expert = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="""
    당신은 보안 전문가입니다.
    - 보안 취약점을 찾습니다
    - 보안 베스트 프랙티스를 적용합니다
    - 위험도를 평가합니다
    - 보안 개선 방안을 제시합니다
    """
)

performance_expert = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="""
    당신은 성능 최적화 전문가입니다.
    - 성능 병목 지점을 찾습니다
    - 최적화 방안을 제시합니다
    - 리소스 사용을 분석합니다
    - 성능 개선 권장사항을 제공합니다
    """
)

print("✅ 전문 Agent들 생성 완료!")


In [ ]:
# 이들을 Subagent로 사용하는 Main Agent 생성
code_review_coordinator = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="""
    당신은 코드 리뷰 조정자입니다.
    코드를 받으면:
    1. 먼저 코드 리뷰어에게 일반적인 코드 품질 검토를 요청
    2. 보안 전문가에게 보안 검토를 요청
    3. 성능 전문가에게 성능 분석을 요청
    4. 모든 피드백을 종합하여 최종 리뷰 보고서를 작성
    
    각 전문가의 의견을 존중하고 통합된 개선 방안을 제시하세요.
    """,
    subagents=[
        {
            "name": "code_reviewer",
            "description": "코드 품질과 베스트 프랙티스를 검토하는 전문가",
            "agent": code_reviewer
        },
        {
            "name": "security_expert", 
            "description": "보안 취약점을 찾고 보안 개선 방안을 제시하는 전문가",
            "agent": security_expert
        },
        {
            "name": "performance_expert",
            "description": "성능 병목과 최적화 방안을 제시하는 전문가",
            "agent": performance_expert
        }
    ]
)

print("✅ Code Review Coordinator 생성 완료!")


## 8. 실습: 종합 코드 리뷰 시스템


In [ ]:
# 리뷰할 샘플 코드
sample_code = '''
def process_user_data(user_input):
    # 사용자 데이터 처리
    query = "SELECT * FROM users WHERE id = " + user_input
    result = execute_query(query)
    
    # 전체 데이터를 메모리에 로드
    all_data = []
    for item in result:
        all_data.append(item)
        # 각 항목마다 복잡한 연산 수행
        for i in range(1000000):
            x = i * i
    
    return all_data

def get_user_password(username):
    # 비밀번호를 평문으로 저장
    passwords = {
        "admin": "admin123",
        "user": "password"
    }
    return passwords.get(username)
'''

review_request = f"""
다음 Python 코드를 종합적으로 리뷰해주세요:

```python
{sample_code}
```

각 전문가의 관점에서 검토하고, 
'code_review_report.txt' 파일에 다음 내용을 포함한 보고서를 작성해주세요:
1. 코드 품질 리뷰
2. 보안 분석
3. 성능 분석
4. 종합 개선 권장사항
5. 우선순위별 액션 아이템
"""

print("🔍 종합 코드 리뷰 시작...\n")

response = code_review_coordinator.invoke(
    {"messages": [{"role": "user", "content": review_request}]},
    config={"configurable": {"thread_id": "code-review-demo"}}
)

print("\n✅ 코드 리뷰 완료!")
print(response["messages"][-1].content)


## 9. 작은 규모 프로젝트 구조 설계하기

실제 프로젝트에서 Subagent를 어떻게 구조화하는지 예제를 살펴봅시다.

### 예제: 블로그 글 생성 시스템


In [ ]:
# 블로그 글 생성 시스템 구조
# 1. Topic Researcher - 주제 조사
topic_researcher = {
    "name": "topic_researcher",
    "description": "주어진 주제에 대해 최신 정보와 트렌드를 조사하는 전문가",
    "system_prompt": """
    당신은 주제 조사 전문가입니다.
    - 주제와 관련된 최신 정보를 검색합니다
    - 인기 있는 키워드와 트렌드를 파악합니다
    - 대상 독자가 관심 있어 할 만한 각도를 찾습니다
    - 조사 결과를 구조화된 형태로 정리합니다
    """
}

# 2. Outline Creator - 개요 작성
outline_creator = {
    "name": "outline_creator",
    "description": "블로그 글의 구조와 개요를 작성하는 전문가",
    "system_prompt": """
    당신은 콘텐츠 구조 전문가입니다.
    - 조사된 정보를 바탕으로 글의 구조를 설계합니다
    - 논리적인 흐름을 만듭니다
    - 각 섹션의 주요 포인트를 정리합니다
    - 독자 친화적인 구조를 제안합니다
    """
}

# 3. Content Writer - 본문 작성
content_writer = {
    "name": "content_writer",
    "description": "매력적인 블로그 본문을 작성하는 전문 작가",
    "system_prompt": """
    당신은 전문 콘텐츠 작가입니다.
    - 개요를 바탕으로 상세한 본문을 작성합니다
    - 흥미롭고 읽기 쉬운 문체를 사용합니다
    - 적절한 예시와 설명을 포함합니다
    - SEO 친화적인 내용을 작성합니다
    """
}

# 4. SEO Optimizer - SEO 최적화
seo_optimizer = {
    "name": "seo_optimizer",
    "description": "검색 엔진 최적화를 담당하는 전문가",
    "system_prompt": """
    당신은 SEO 전문가입니다.
    - 적절한 키워드를 배치합니다
    - 메타 설명을 작성합니다
    - 제목 태그를 최적화합니다
    - 내부/외부 링크를 제안합니다
    """
}

# 5. Editor - 최종 편집
editor = {
    "name": "editor",
    "description": "최종 검토와 편집을 담당하는 에디터",
    "system_prompt": """
    당신은 베테랑 편집자입니다.
    - 문법과 맞춤법을 검토합니다
    - 가독성을 개선합니다
    - 일관성을 확인합니다
    - 최종 품질을 보증합니다
    """
}

print("✅ 블로그 생성 시스템 Subagent 정의 완료!")


In [ ]:
# 블로그 생성 조정자 Agent
blog_generator = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="""
    당신은 블로그 콘텐츠 프로덕션 매니저입니다.
    
    블로그 글 생성 워크플로우:
    1. topic_researcher에게 주제 조사 요청
    2. outline_creator에게 조사 결과를 바탕으로 개요 작성 요청
    3. content_writer에게 개요를 바탕으로 본문 작성 요청
    4. seo_optimizer에게 SEO 최적화 요청
    5. editor에게 최종 검토 요청
    6. 모든 결과를 통합하여 완성된 블로그 글을 파일로 저장
    
    각 단계를 순차적으로 진행하고, 중간 결과를 다음 단계에 전달하세요.
    """,
    subagents=[topic_researcher, outline_creator, content_writer, seo_optimizer, editor]
)

print("✅ Blog Generator 생성 완료!")


In [ ]:
# 블로그 글 생성 실행
blog_request = """
다음 주제로 블로그 글을 작성해주세요:

주제: "AI Agents의 실제 비즈니스 활용 사례"

요구사항:
- 대상 독자: 기업의 기술 의사결정자
- 길이: 약 1500-2000 단어
- 톤: 전문적이면서 접근하기 쉽게
- 실제 사례 3-4개 포함

최종 결과를 'blog_ai_agents.md' 파일로 저장해주세요.
"""

print("📝 블로그 글 생성 워크플로우 시작...\n")
print("="*60)

# 스트리밍으로 진행 과정 확인
for chunk in blog_generator.stream(
    {"messages": [{"role": "user", "content": blog_request}]},
    config={"configurable": {"thread_id": "blog-generation"}},
    stream_mode="updates"
):
    for node_name, node_output in chunk.items():
        if node_name != "__start__":
            print(f"\n📍 {node_name} 처리 중...")

print("\n" + "="*60)
print("\n✅ 블로그 글 생성 완료!")


## 10. 주요 개념 정리

### Subagent 정의 방법 비교

#### 1. Dictionary-based Subagent
```python
{
    "name": "subagent_name",
    "description": "what this subagent does",
    "system_prompt": "detailed instructions"
}
```
**장점**: 간단하고 빠르게 정의 가능  
**단점**: 제한된 커스터마이징

#### 2. CompiledSubAgent
```python
{
    "name": "subagent_name",
    "description": "what this subagent does",
    "agent": create_agent(...)  # 미리 생성된 Agent
}
```
**장점**: 완전한 제어, 복잡한 설정 가능  
**단점**: 더 많은 코드 필요

### Best Practices

1. **단일 책임 원칙**: 각 Subagent는 하나의 명확한 역할만 담당
2. **명확한 인터페이스**: description을 통해 Subagent의 역할을 명확히 정의
3. **순차적 처리**: Main Agent가 Subagent 간 작업 흐름을 관리
4. **결과 통합**: Main Agent가 각 Subagent의 결과를 취합


## 11. 다음 단계

이제 Subagent와 Workflow 구조화 방법을 이해했습니다!

다음 강의에서는:
- ✅ **Backend & Persistent Storage**: 상태를 영구적으로 저장하고 관리하는 방법
- ✅ **Middleware & 고급 설정**: 더욱 세밀한 제어와 커스터마이징

를 배우게 됩니다.

---

## 💡 연습 문제

### 문제 1: 뉴스 요약 시스템
다음 Subagent들을 가진 뉴스 요약 시스템을 만들어보세요:
- `news_crawler`: 뉴스 기사 수집
- `summarizer`: 기사 요약
- `categorizer`: 카테고리 분류
- `aggregator`: 최종 뉴스 다이제스트 생성

### 문제 2: 데이터 파이프라인
데이터 수집 → 정제 → 분석 → 시각화 → 보고서 생성 파이프라인을 Subagent로 구현해보세요.

### 문제 3: 다국어 번역 시스템
원문 분석 → 번역 → 현지화 → 품질 검토의 워크플로우를 가진 번역 시스템을 만들어보세요.

---

## 📚 참고 자료
- [Deep Agents Subagents 문서](https://docs.langchain.com/oss/python/deepagents/subagents)
- [LangGraph Subgraphs](https://langchain-ai.github.io/langgraph/concepts/low_level/#subgraphs)
- [Agent Architectures](https://python.langchain.com/docs/concepts/architecture/)
